# H2-PRISM Baseline Models Evaluation (Person 2: Faisal)
Compares Logistic Regression, Random Forest, and XGBoost using labeled windows from the selected H2-SimNet scenarios.

## 1- Imports

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier

from src.data import load_scenario
from src.features import extract_window_features
from src.fingerprint import extract_transient_fingerprint
from src.physics import compute_mass_balance_residual
from src.propagation import estimate_propagation_delay

## 2- Load actual scenarios and create labeled windows

Windows are split at the sample level, so results measure performance on these recordings and should not be interpreted as generalization to unseen scenarios.

In [7]:
project_root = Path.cwd()
if not (project_root / "data" / "processed").is_dir():
    project_root = project_root.parent

data_dir = project_root / "data" / "processed"
scenario_files = sorted(data_dir.glob("*.csv"))
if not scenario_files:
    raise FileNotFoundError(f"No processed scenario CSVs found in {data_dir}")

window_size = 100
required_columns = {
    "timestamp", "p_upstream", "p_downstream",
    "m_inlet", "m_outlet", "label",
}
scenario_features = []

for scenario_path in scenario_files:
    scenario = load_scenario(scenario_path)
    missing_columns = required_columns.difference(scenario.columns)
    if missing_columns:
        raise ValueError(
            f"{scenario_path.name} is missing columns: {sorted(missing_columns)}"
        )

    for start in range(0, len(scenario) - window_size + 1, window_size):
        window = scenario.iloc[start : start + window_size]
        window_labels = window["label"].dropna().astype(str)
        if window_labels.empty:
            continue

        features = extract_window_features(window)
        features["scenario_id"] = scenario_path.stem
        features["window_start"] = start
        features["label"] = window_labels.value_counts().idxmax()
        scenario_features.append(features)

feature_table = pd.DataFrame(scenario_features)
if feature_table.empty:
    raise ValueError("No labeled windows were created from the processed scenarios.")

print(f"Scenarios loaded: {len(scenario_files)}")
print(f"Labeled windows: {len(feature_table)}")
print(feature_table["label"].value_counts().sort_index())

Scenarios loaded: 6
Labeled windows: 1250
label
COMPRESSOR_DELAY     10
COMPRESSOR_FAULT    200
LEAK_OVERLAPPED     196
LEAK_STRONG_G3      199
LEAK_WEAK_G4        196
NORMAL              449
Name: count, dtype: int64


In [8]:
excluded_columns = {"label", "scenario_id", "window_start"}

feature_names = [
    column
    for column in feature_table.columns
    if column not in excluded_columns
]

feature_table[feature_names] = (
    feature_table[feature_names]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

train_indices, test_indices = train_test_split(
    feature_table.index,
    test_size=0.25,
    random_state=42,
    stratify=feature_table["label"],
)

train_table = feature_table.loc[train_indices].copy()
test_table = feature_table.loc[test_indices].copy()

X_train = train_table[feature_names]
X_test = test_table[feature_names]

y_train = train_table["label"]
y_test = test_table["label"]

## 3- Train and compare baseline models

In [9]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42,
        ),
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
}

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

models["XGBoost"] = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    eval_metric="mlogloss",
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
)

evaluation_rows = []
for model_name, model in models.items():
    if model_name == "XGBoost":
        model.fit(X_train, y_train_encoded)
        predictions_encoded = model.predict(X_test)
        predictions = label_encoder.inverse_transform(
            predictions_encoded.astype(int)
        )
    else:
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

    balanced_accuracy = balanced_accuracy_score(y_test, predictions)
    evaluation_rows.append(
        {"model": model_name, "balanced_accuracy": balanced_accuracy}
    )

    print(f"\n{model_name} | balanced accuracy: {balanced_accuracy:.3f}")
    print(classification_report(y_test, predictions, zero_division=0))

evaluation_table = pd.DataFrame(evaluation_rows).sort_values(
    "balanced_accuracy",
    ascending=False,
    ignore_index=True,
)
display(evaluation_table)


Logistic Regression | balanced accuracy: 0.786
                  precision    recall  f1-score   support

COMPRESSOR_DELAY       1.00      1.00      1.00         3
COMPRESSOR_FAULT       1.00      1.00      1.00        50
 LEAK_OVERLAPPED       1.00      1.00      1.00        49
  LEAK_STRONG_G3       0.39      0.36      0.38        50
    LEAK_WEAK_G4       0.38      0.78      0.51        49
          NORMAL       0.98      0.58      0.73       112

        accuracy                           0.71       313
       macro avg       0.79      0.79      0.77       313
    weighted avg       0.80      0.71      0.73       313


Random Forest | balanced accuracy: 1.000
                  precision    recall  f1-score   support

COMPRESSOR_DELAY       1.00      1.00      1.00         3
COMPRESSOR_FAULT       1.00      1.00      1.00        50
 LEAK_OVERLAPPED       1.00      1.00      1.00        49
  LEAK_STRONG_G3       1.00      1.00      1.00        50
    LEAK_WEAK_G4       1.00      1.0

,model,balanced_accuracy
0,Random Forest,1.000000
1,XGBoost,0.996667
2,Logistic Regression,0.785978
